In [1]:
import platform
import psutil
from typing import Tuple, Union
from timeit import timeit
from warnings import warn

# PyTorch dependencies
import torch
import torch.backends.opt_einsum as opt_einsum
from torch import Tensor

# Internal dependencies
from thoad import backward, Controller

In [2]:
# control size of tensors
TENSOR_SCALE: Union[int, float] = 1
REPEAT_SCALE: Union[int, float] = 1

In [3]:
sys: platform.uname_result = platform.uname()
print(f"system           {sys.system} {sys.release} {sys.version}")

system           Windows 11 10.0.22631


In [4]:
dev: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if dev.type == 'cuda':
    idx = dev.index if dev.index is not None else 0
    props: "_CudaDeviceProperties" = torch.cuda.get_device_properties(idx)
    name: str = props.name
    total_mem_gb: float = props.total_memory / (1024**3)
    print(f"using device     {dev} -> {name}")
    print(f"device memory    {total_mem_gb:.1f} GB)")
else:
    cpu_name: str = platform.processor() or "CPU"
    print(f"using device     {dev} -> {cpu_name}")
    print(f"physical cores   {psutil.cpu_count(logical=False)}")
    print(f"logical cores    {psutil.cpu_count(logical=True)}")

using device     cuda -> NVIDIA GeForce RTX 4070 Ti
device memory    12.0 GB)


In [5]:
if opt_einsum.is_available():
    opt_einsum.enabled = True
    opt_einsum.strategy = "greedy"
    print("opt_einsum backend enabled")
else:
    warn(
        "opt_einsum backend is not available. "
        "For better performance, install and enable opt_einsum.",
        UserWarning
    )

opt_einsum backend enabled


definition of MLP

In [6]:
def foward_pass(X: Tensor, *params) -> Tensor:
    T: Tensor = X
    for i, P in enumerate(params):
        last_step: bool = i == (len(params) - 1)
        T = T @ P
        T = torch.softmax(T, dim=1) if last_step else torch.relu(T)
    return T.sum()

## **Benchmark jacobians on full MLP**

definition of helper functions to meassure jacobian times

In [7]:
def time_autograd_jacobian(param_grad: bool, reps: int, X: Tensor, *params) -> float:
    def _fixed_forward_pass(X) -> Tensor:
        return foward_pass(X, *params)
    def _foward_and_backward() -> None:
        if param_grad:
            torch.autograd.functional.jacobian(func=foward_pass, inputs=(X, *params))
        else:
            torch.autograd.functional.jacobian(func=_fixed_forward_pass, inputs=X)
        return None
    time: float = timeit(
        lambda: _foward_and_backward(),
        number=reps,
    )
    return time

def time_thoad_jacobian(param_grad: bool, reps: int, X: Tensor, *params) -> float:
    X.requires_grad_(True)
    params: list[Tensor] = [P.requires_grad_(param_grad) for P in params]
    def _foward_and_backward() -> None:
        T: Tensor = foward_pass(X, *params)
        ctrl: Controller = backward(tensor=T, order=1, crossings=param_grad, keep_batch=True)
        ctrl.clear()
        return None
    time: float = timeit(
        lambda: _foward_and_backward(),
        number=reps,
    )
    return time

**jacobian** computational cost w.r.t. **batch size**

In [8]:
for batch_size in [10, 20, 30, 40, 50, 60, 70, 80]:
    param_size: int = int(10 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(3)]

    reps: int = int(200 * (1/batch_size) * REPEAT_SCALE)
    autograd_time: float = time_autograd_jacobian(False, reps, X, *params)
    thoad_time: float = time_thoad_jacobian(False, reps, X, *params)

    print(
        f"batch size: {batch_size:03d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

batch size: 010 -> autograd: 0.0050  thoad: 0.0055
batch size: 020 -> autograd: 0.0012  thoad: 0.0046
batch size: 030 -> autograd: 0.0006  thoad: 0.0043
batch size: 040 -> autograd: 0.0006  thoad: 0.0048
batch size: 050 -> autograd: 0.0007  thoad: 0.0043
batch size: 060 -> autograd: 0.0006  thoad: 0.0053
batch size: 070 -> autograd: 0.0008  thoad: 0.0052
batch size: 080 -> autograd: 0.0007  thoad: 0.0048


**jacobian** computational cost w.r.t. **param size**

In [9]:
for param_size in [10, 20, 30, 40, 50, 60, 70, 80]:
    batch_size: int = int(10 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(3)]

    reps: int = int(200 * (1/param_size) * REPEAT_SCALE)
    autograd_time: float = time_autograd_jacobian(False, reps, X, *params)
    thoad_time: float = time_thoad_jacobian(False, reps, X, *params)

    print(
        f"param size: {param_size:03d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

param size: 010 -> autograd: 0.0005  thoad: 0.0054
param size: 020 -> autograd: 0.0006  thoad: 0.0057
param size: 030 -> autograd: 0.0006  thoad: 0.0057
param size: 040 -> autograd: 0.0006  thoad: 0.0053
param size: 050 -> autograd: 0.0006  thoad: 0.0051
param size: 060 -> autograd: 0.0006  thoad: 0.0056
param size: 070 -> autograd: 0.0005  thoad: 0.0049
param size: 080 -> autograd: 0.0006  thoad: 0.0055


**jacobian** computational cost w.r.t. **graph depth**

In [10]:
for depth in range(2, 21):
    batch_size: int = 60
    param_size: int = int(20 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(depth)]

    reps: int = int(200 * (1/depth) * REPEAT_SCALE)
    autograd_time: float = time_autograd_jacobian(False, reps, X, *params)
    thoad_time: float = time_thoad_jacobian(False, reps, X, *params)

    print(
        f"graph depth: {depth:02d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

graph depth: 02 -> autograd: 0.0005  thoad: 0.0037
graph depth: 03 -> autograd: 0.0006  thoad: 0.0047
graph depth: 04 -> autograd: 0.0006  thoad: 0.0056
graph depth: 05 -> autograd: 0.0006  thoad: 0.0066
graph depth: 06 -> autograd: 0.0009  thoad: 0.0077
graph depth: 07 -> autograd: 0.0010  thoad: 0.0085
graph depth: 08 -> autograd: 0.0011  thoad: 0.0094
graph depth: 09 -> autograd: 0.0012  thoad: 0.0102
graph depth: 10 -> autograd: 0.0013  thoad: 0.0111
graph depth: 11 -> autograd: 0.0014  thoad: 0.0124
graph depth: 12 -> autograd: 0.0011  thoad: 0.0128
graph depth: 13 -> autograd: 0.0010  thoad: 0.0140
graph depth: 14 -> autograd: 0.0014  thoad: 0.0162
graph depth: 15 -> autograd: 0.0019  thoad: 0.0164
graph depth: 16 -> autograd: 0.0016  thoad: 0.0170
graph depth: 17 -> autograd: 0.0020  thoad: 0.0184
graph depth: 18 -> autograd: 0.0020  thoad: 0.0199
graph depth: 19 -> autograd: 0.0021  thoad: 0.0211
graph depth: 20 -> autograd: 0.0022  thoad: 0.0222


**jacobian** computational cost w.r.t. **batch size** (param gradients included)

In [11]:
for batch_size in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]:
    param_size: int = int(10 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(3)]

    reps: int = int(200 * (1/batch_size) * REPEAT_SCALE)
    autograd_time: float = time_autograd_jacobian(True, reps, X, *params)
    thoad_time: float = time_thoad_jacobian(True, reps, X, *params)

    print(
        f"batch size: {batch_size:02d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

batch size: 05 -> autograd: 0.0008  thoad: 0.0085
batch size: 10 -> autograd: 0.0008  thoad: 0.0079
batch size: 15 -> autograd: 0.0009  thoad: 0.0079
batch size: 20 -> autograd: 0.0009  thoad: 0.0081
batch size: 25 -> autograd: 0.0008  thoad: 0.0083
batch size: 30 -> autograd: 0.0008  thoad: 0.0101
batch size: 35 -> autograd: 0.0012  thoad: 0.0083
batch size: 40 -> autograd: 0.0008  thoad: 0.0076
batch size: 45 -> autograd: 0.0006  thoad: 0.0084
batch size: 50 -> autograd: 0.0008  thoad: 0.0080


**jacobian** computational cost w.r.t. **param size** (param gradients included)

In [12]:
for param_size in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]:
    batch_size: int = int(10 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(3)]

    reps: int = int(200 * (1/param_size) * REPEAT_SCALE)
    autograd_time: float = time_autograd_jacobian(True, reps, X, *params)
    thoad_time: float = time_thoad_jacobian(True, reps, X, *params)

    print(
        f"param size: {param_size:02d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

param size: 05 -> autograd: 0.0007  thoad: 0.0084
param size: 10 -> autograd: 0.0006  thoad: 0.0080
param size: 15 -> autograd: 0.0009  thoad: 0.0079
param size: 20 -> autograd: 0.0006  thoad: 0.0081
param size: 25 -> autograd: 0.0007  thoad: 0.0090
param size: 30 -> autograd: 0.0007  thoad: 0.0084
param size: 35 -> autograd: 0.0007  thoad: 0.0086
param size: 40 -> autograd: 0.0006  thoad: 0.0077
param size: 45 -> autograd: 0.0007  thoad: 0.0078
param size: 50 -> autograd: 0.0006  thoad: 0.0078


**jacobian** computational cost w.r.t. **graph depth** (param gradients included)

In [13]:
for depth in range(2, 7):
    batch_size: int = 10
    param_size: int = int(10 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(depth)]

    reps: int = int(200 * (1/depth) * REPEAT_SCALE)
    autograd_time: float = time_autograd_jacobian(True, reps, X, *params)
    thoad_time: float = time_thoad_jacobian(True, reps, X, *params)

    print(
        f"graph depth: {depth:02d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

graph depth: 02 -> autograd: 0.0004  thoad: 0.0058
graph depth: 03 -> autograd: 0.0006  thoad: 0.0080
graph depth: 04 -> autograd: 0.0008  thoad: 0.0103
graph depth: 05 -> autograd: 0.0006  thoad: 0.0133
graph depth: 06 -> autograd: 0.0010  thoad: 0.0153


## **Benchmark hessians on full MLP**

definition of helper functions to meassure hessian times

In [14]:
def time_autograd_hessian(param_grad: bool, reps: int, X: Tensor, *params) -> float:
    def _fixed_forward_pass(X) -> Tensor:
        return foward_pass(X, *params)
    def _foward_and_backward() -> None:
        if param_grad:
            torch.autograd.functional.hessian(func=foward_pass, inputs=(X, *params))
        else:
            torch.autograd.functional.hessian(func=_fixed_forward_pass, inputs=X)
        return None
    time: float = timeit(
        lambda: _foward_and_backward(),
        number=reps,
    )
    return time

def time_thoad_hessian(param_grad: bool, reps: int, X: Tensor, *params) -> float:
    X.requires_grad_(True)
    params: list[Tensor] = [P.requires_grad_(param_grad) for P in params]
    def _foward_and_backward() -> None:
        T: Tensor = foward_pass(X, *params)
        ctrl: Controller = backward(tensor=T, order=2, crossings=param_grad, keep_batch=True)
        ctrl.clear()
        return None
    time: float = timeit(
        lambda: _foward_and_backward(),
        number=reps,
    )
    return time

**hessian** computational cost w.r.t. **batch size**

In [15]:
for batch_size in [10, 20, 30, 40, 50, 60, 70, 80]:
    param_size: int = int(10 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(3)]

    reps: int = int(200 * (1/batch_size) * REPEAT_SCALE)
    autograd_time: float = time_autograd_hessian(False, reps, X, *params)
    thoad_time: float = time_thoad_hessian(False, reps, X, *params)

    print(
        f"batch size: {batch_size:03d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

batch size: 010 -> autograd: 0.0697  thoad: 0.0090
batch size: 020 -> autograd: 0.1539  thoad: 0.0083
batch size: 030 -> autograd: 0.1828  thoad: 0.0083
batch size: 040 -> autograd: 0.2486  thoad: 0.0081
batch size: 050 -> autograd: 0.2854  thoad: 0.0088
batch size: 060 -> autograd: 0.2794  thoad: 0.0088
batch size: 070 -> autograd: 0.3318  thoad: 0.0083
batch size: 080 -> autograd: 0.4206  thoad: 0.0081


**hessian** computational cost w.r.t. **param size**

In [16]:
for param_size in [10, 20, 30, 40, 50, 60, 70, 80]:
    batch_size: int = int(10 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(3)]

    reps: int = int(200 * (1/param_size) * REPEAT_SCALE)
    autograd_time: float = time_autograd_hessian(False, reps, X, *params)
    thoad_time: float = time_thoad_hessian(False, reps, X, *params)

    print(
        f"param size: {param_size:02d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

param size: 10 -> autograd: 0.0467  thoad: 0.0084
param size: 20 -> autograd: 0.0883  thoad: 0.0079
param size: 30 -> autograd: 0.1336  thoad: 0.0084
param size: 40 -> autograd: 0.1992  thoad: 0.0083
param size: 50 -> autograd: 0.2633  thoad: 0.0092
param size: 60 -> autograd: 0.3726  thoad: 0.0080
param size: 70 -> autograd: 0.3603  thoad: 0.0094
param size: 80 -> autograd: 0.4317  thoad: 0.0099


**hessian** computational cost w.r.t. **graph depth**

In [17]:
for depth in range(2, 11):
    input_size: int = 20
    param_size: int = int(20 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(depth)]

    reps: int = int(200 * (1/depth) * REPEAT_SCALE)
    autograd_time: float = time_autograd_hessian(False, reps, X, *params)
    thoad_time: float = time_thoad_hessian(False, reps, X, *params)

    print(
        f"graph depth: {depth:02d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

graph depth: 02 -> autograd: 0.0876  thoad: 0.0065
graph depth: 03 -> autograd: 0.1151  thoad: 0.0083
graph depth: 04 -> autograd: 0.1472  thoad: 0.0098
graph depth: 05 -> autograd: 0.1231  thoad: 0.0122
graph depth: 06 -> autograd: 0.1397  thoad: 0.0145
graph depth: 07 -> autograd: 0.1445  thoad: 0.0171
graph depth: 08 -> autograd: 0.1574  thoad: 0.0187
graph depth: 09 -> autograd: 0.2402  thoad: 0.0203
graph depth: 10 -> autograd: 0.1929  thoad: 0.0220


**hessian** computational cost w.r.t. **batch size** (param gradients included)

In [18]:
for batch_size in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]:
    param_size: int = int(10 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(3)]

    reps: int = int(100 * (1/batch_size) * REPEAT_SCALE)
    autograd_time: float = time_autograd_hessian(True, reps, X, *params)
    thoad_time: float = time_thoad_hessian(True, reps, X, *params)

    print(
        f"batch size: {batch_size:02d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

batch size: 05 -> autograd: 0.3046  thoad: 0.0236
batch size: 10 -> autograd: 0.2533  thoad: 0.0251
batch size: 15 -> autograd: 0.2632  thoad: 0.0273
batch size: 20 -> autograd: 0.4310  thoad: 0.0258
batch size: 25 -> autograd: 0.4767  thoad: 0.0238
batch size: 30 -> autograd: 0.4594  thoad: 0.0221
batch size: 35 -> autograd: 0.5253  thoad: 0.0263
batch size: 40 -> autograd: 0.4286  thoad: 0.0264
batch size: 45 -> autograd: 0.4517  thoad: 0.0245
batch size: 50 -> autograd: 0.4816  thoad: 0.0248


**hessian** computational cost w.r.t. **param size** (param gradients included)

In [19]:
for param_size in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]:
    batch_size: int = int(10 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(3)]

    reps: int = int(100 * (1/param_size) * REPEAT_SCALE)
    autograd_time: float = time_autograd_hessian(True, reps, X, *params)
    thoad_time: float = time_thoad_hessian(True, reps, X, *params)

    print(
        f"param size: {param_size:02d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

param size: 05 -> autograd: 0.0719  thoad: 0.0249
param size: 10 -> autograd: 0.2915  thoad: 0.0276
param size: 15 -> autograd: 0.5951  thoad: 0.0253
param size: 20 -> autograd: 0.7527  thoad: 0.0228
param size: 25 -> autograd: 1.4672  thoad: 0.0234
param size: 30 -> autograd: 1.9630  thoad: 0.0234
param size: 35 -> autograd: 3.6257  thoad: 0.0221
param size: 40 -> autograd: 3.5508  thoad: 0.0240
param size: 45 -> autograd: 4.6980  thoad: 0.0231
param size: 50 -> autograd: 5.5392  thoad: 0.0251


**hessian** computational cost w.r.t. **graph depth** (param gradients included)

In [20]:
for depth in [2, 3, 4, 5, 6, 7, 8]:
    batch_size: int = 10
    param_size: int = int(10 * TENSOR_SCALE)
    x_shape: Tuple[int, int] = (batch_size, param_size)
    p_shape: Tuple[int, int] = (param_size, param_size)

    X: Tensor = torch.rand(size=x_shape, device=dev)
    params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(depth)]

    reps: int = int(100 * (1/depth) * REPEAT_SCALE)
    autograd_time: float = time_autograd_hessian(True, reps, X, *params)
    thoad_time: float = time_thoad_hessian(True, reps, X, *params)

    print(
        f"graph depth: {depth:02d} -> "
        f"autograd: {autograd_time / reps:.4f}  thoad: {thoad_time / reps:.4f}"
    )

graph depth: 02 -> autograd: 0.1699  thoad: 0.0154
graph depth: 03 -> autograd: 0.2698  thoad: 0.0239
graph depth: 04 -> autograd: 0.4218  thoad: 0.0348
graph depth: 05 -> autograd: 0.6269  thoad: 0.0520
graph depth: 06 -> autograd: 0.6430  thoad: 0.0713
graph depth: 07 -> autograd: 0.8042  thoad: 0.0928
graph depth: 08 -> autograd: 1.1500  thoad: 0.1193
